# 02. An?lisis de mercado

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "funciones_reservas.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("No se encontr? la ra?z del proyecto.")
sys.path.insert(0, str(ROOT / "src"))
PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
from funciones_reservas import (leer_snapshot, snapshot_es_valido,
                                resumen_emision_anual, exportar_tabla,
                                exportar_figura)

emision_snapshot = PROCESSED / "emision_ajustada_2025.parquet"
siniestros_snapshot = PROCESSED / "siniestros_ajustados_2025.parquet"
if not snapshot_es_valido(emision_snapshot, periodo_valuacion="2025-12", archivos_origen=[ROOT / "data" / "raw" / "2026 BASE RESERVAS.xlsx"]):
    raise FileNotFoundError("El snapshot de emisi?n no existe o est? desactualizado. Ejecuta 01_actualizacion_inpc.ipynb con el INPC disponible.")
if not snapshot_es_valido(siniestros_snapshot, periodo_valuacion="2025-12", archivos_origen=[ROOT / "data" / "raw" / "2026 BASE RESERVAS.xlsx"]):
    raise FileNotFoundError("El snapshot de siniestros no existe o est? desactualizado. Ejecuta 01_actualizacion_inpc.ipynb con el INPC disponible.")
emision_ajustada = leer_snapshot(emision_snapshot)
siniestros_ajustados = leer_snapshot(siniestros_snapshot)


## Evoluci?n de primas, retenci?n y comisiones

In [ ]:
columnas_monetarias = ["prima_emitida_ajustada", "prima_retenida_ajustada", "comision_directa_ajustada"]
emision_anual = resumen_emision_anual(emision_ajustada)
exportar_tabla(emision_anual, REPORTS / "tables" / "emision_anual.csv")
comparacion = emision_anual.set_index("a?o")[["prima_emitida_ajustada", "prima_retenida_ajustada", "comisiones_pct"]]
base = comparacion.loc[2015]
comparacion = comparacion.div(base).mul(100).reset_index()
fig = px.line(comparacion, x="a?o", y=comparacion.columns[1:], markers=True,
              title="Evoluci?n relativa de primas y comisiones",
              labels={"a?o": "A?o", "value": "?ndice (2015=100)", "variable": "Indicador"})
fig.update_layout(hovermode="x unified", legend_title_text="Indicador")
exportar_figura(fig, REPORTS / "figures" / "evolucion_primas_comisiones")
fig.show()


Con fines comparativos, se normalizaron las variables tomando 2015 como año base (índice de 100), para observar sus cambios relativos y facilitar la comparación de los comportamientos a lo largo de los años.

Se observa una tendencia creciente en las primas emitidas, alcanzando un aumento aproximado del 150% en 2025 respecto a 2015 lo cual el aumento de este último año puede deberse a los conflictos entre Estados Unidos e Irán. También destaca una caída importante en 2020, posiblemente relacionada con la disminución de la actividad marítima durante la pandemia, lo que pudo afectar la contratación de seguros. 

Por otro lado, las primas retenidas se mantienen relativamente estables a pesar del crecimiento de las emitidas, esto podría deberse a una mayor participación del reaseguro, lo que implicaría una menor proporción de riesgos asumidos directamente por las aseguradoras. 

Finalmente, la proporción de comisiones aumentó durante 2020 y 2021, posiblemente porque las primas emitidas disminuyeron más que las comisiones, mientras que en los últimos años esta proporción se redujo, lo que sugiere un menor peso relativo de las comisiones dentro del mercado.

## Distribuci?n de primas por tipo de embarcaci?n

In [ ]:
participacion = (emision_ajustada.groupby(["a?o", "tipo_embarcacion"])["prima_emitida_ajustada"].sum().unstack(fill_value=0))
principales = participacion.sum().nlargest(6).index
participacion["Otros"] = participacion.drop(columns=principales).sum(axis=1)
participacion = participacion[list(principales) + ["Otros"]]
totales = participacion.sum(axis=1)
if (totales <= 0).any():
    raise ValueError("La participaci?n no puede calcularse para a?os con emisi?n neta no positiva.")
participacion = participacion.div(totales, axis=0).mul(100).reset_index().melt(
    id_vars="a?o", var_name="Tipo de embarcaci?n", value_name="Participaci?n (%)")
exportar_tabla(participacion, REPORTS / "tables" / "participacion_primas_embarcacion.csv")
fig = px.area(participacion, x="a?o", y="Participaci?n (%)", color="Tipo de embarcaci?n",
              title="Distribuci?n anual de primas por tipo de embarcaci?n")
fig.update_layout(yaxis=dict(range=[0, 100], ticksuffix="%"), xaxis_title="A?o", hovermode="x unified")
exportar_figura(fig, REPORTS / "figures" / "participacion_primas_embarcacion")
fig.show()


Con el objetivo de analizar cómo ha cambiado la composición del mercado, se calculó la participación porcentual de cada tipo de embarcación respecto a la prima emitida total de cada año.

Se observa que entre 2015 y 2022 los abastecedores tuvieron una participación importante, destacando especialmente en 2019, sin embargo, a partir de 2023 los buques tanque petroleros comenzaron a dominar el mercado, alcanzando aproximadamente el 77% de las primas emitidas en 2025. Este cambio podría estar relacionado con lo antes ya mensionado de las tensiónes entre Estados Unidos e Irán, con ello aumento de los riesgos en el transporte marítimo de petróleo, que han provocado incrementos en los costos de aseguramiento de ciertas rutas, lo que podría explicar una mayor concentración de primas en este tipo de embarcaciones.

Esto sugiere que el mercado se ha vuelto más dependiente del segmento petrolero, lo que podría representar una mayor exposición a los riesgos asociados con esta actividad y una menor diversificación de las primas emitidas.

## Distribuci?n de siniestros por causa

In [ ]:
columnas_pagos = [f"{i}_monto pagado_ajustada" for i in range(4)]
siniestros_analisis = siniestros_ajustados.copy()
siniestros_analisis["monto_total"] = siniestros_analisis[columnas_pagos].sum(axis=1, min_count=1)
siniestros_anual = (siniestros_analisis.groupby(["a?o", "causa_siniestro"], as_index=False)["monto_total"].sum().sort_values("a?o"))
exportar_tabla(siniestros_anual, REPORTS / "tables" / "siniestros_anuales_por_causa.csv")
fig = px.bar(siniestros_anual, x="a?o", y="monto_total", color="causa_siniestro", barmode="relative",
             title="Distribuci?n anual de siniestros por causa",
             labels={"a?o": "A?o de origen", "monto_total": "Monto ajustado (MXN)", "causa_siniestro": "Causa"})
fig.update_layout(xaxis=dict(dtick=1), yaxis=dict(tickformat=",.0f"), hovermode="x unified", height=600)
exportar_figura(fig, REPORTS / "figures" / "siniestros_por_causa")
fig.show()


Se observa que 2015 presenta el mayor monto de siniestros del periodo, con aproximadamente 4,200 millones de pesos, de los cuales al rededor de 3,800 millones son principalmente en fenómenos naturales, esto podría relacionarse con la intensa temporada de huracanes registrada ese año, especialmente con el paso del huracán Patricia, lo que refleja la exposición del mercado marítimo a eventos naturales capaces de generar pérdidas importantes. Posteriormente, los montos disminuyen, aunque vuelven a presentar incrementos en 2023 y 2024, con una mayor participación de distintas causas, como varaduras y encallamientos, lo que muestra que las pérdidas del mercado también dependen de riesgos operativos y no únicamente de fenómenos naturales.

## Conclusi?n del an?lisis de mercado

## 2.3 Conclusión del Análisis de Mercado 

Se observa que el mercado de seguros de embarcaciones presenta un crecimiento general en las primas emitidas, aunque con fluctuaciones importantes, como la caída de 2020, posiblemente relacionada con la disminución de la actividad marítima durante la pandemia, lo que pudo afectar la contratación de seguros. Por otro lado, las primas se han concentrado cada vez más en los buques tanque petroleros, posiblemente debido a una mayor actividad o al incremento de los costos de aseguramiento en este sector por conflictos geopolíticos, lo que implica una mayor dependencia del mercado hacia este tipo de embarcaciones.

En cuanto a los siniestros, destaca 2015 por las elevadas pérdidas asociadas a fenómenos naturales, posiblemente relacionadas con la temporada de huracanes, especialemente el paso del huracán Patricia, mientras que en años posteriores se observa una mayor participación de causas operativas, como varaduras y encallamientos. 

En conjunto, los resultados muestran un mercado en crecimiento, pero con una importante concentración de primas y exposición a pérdidas extraordinarias, lo que resalta la importancia de una adecuada diversificación y gestión de riesgos.

Las explicaciones sobre conflictos, pandemia y huracanes se conservan como hip?tesis no verificadas, no como causalidades demostradas.